<a href="https://colab.research.google.com/github/the-name-is-madhu/AI-lab/blob/main/WEEK8_Forward_Chaining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import re

class Rule:
    def __init__(self, premises, conclusion):
        self.premises = premises
        self.conclusion = conclusion

    def __repr__(self):
        premises_str = " ^ ".join([f"{p}({','.join(args)})" for p, args in self.premises])
        return f"{premises_str} => {self.conclusion}[0]({','.join(self.conclusion)[1]})"

# --- Core Helper Functions (FIXED) ---

def is_variable(x):
    """Variables must start with a lowercase letter (handles 'x', 'x1', 'y2', etc.)"""
    return isinstance(x, str) and len(x) > 0 and x[0].islower()

def standardize_variables(rule, counter):
    """Renames all variables in a rule to unique ones (e.g., x -> x1) to avoid overlap."""
    subst = {}
    new_premises = []

    for pred, args in rule.premises + [rule.conclusion]:
        for arg in args:
            if is_variable(arg) and arg not in subst:
                subst[arg] = f"{arg}{counter}"
                counter += 1

    for pred, args in rule.premises:
        new_premises.append((pred, [subst.get(a, a) for a in args]))

    c_pred, c_args = rule.conclusion
    new_conclusion = (c_pred, [subst.get(a, a) for a in c_args])

    return Rule(new_premises, new_conclusion), counter

def subst_predicate(theta, predicate):
    """Applies substitution rules to a predicate's arguments."""
    pred_name, args = predicate
    new_args = []
    for arg in args:
        val = arg
        while val in theta:
            val = theta[val]
        new_args.append(val)
    return (pred_name, new_args)

# --- Unification Algorithm (FIXED) ---

def unify(x, y, theta=None):
    if theta is None:
        theta = {}
    if theta is False:
        return False
    elif x == y:
        return theta
    elif is_variable(x):
        return unify_var(x, y, theta)
    elif is_variable(y):
        return unify_var(y, x, theta)
    elif isinstance(x, tuple) and isinstance(y, tuple):
        # Unpack predicate name and argument list properly
        if x[0] != y[0] or len(x[1]) != len(y[1]):
            return False
        return unify(x[1], y[1], theta)
    elif isinstance(x, list) and isinstance(y, list):
        if len(x) != len(y):
            return False
        if not x:
            return theta
        # Unify first elements, then carry the new theta forward to the rest
        new_theta = unify(x[0], y[0], theta)
        return unify(x[1:], y[1:], new_theta)
    else:
        return False

def unify_var(var, val, theta):
    if var in theta:
        return unify(theta[var], val, theta)
    elif val in theta:
        return unify(var, theta[val], theta)
    new_theta = theta.copy()
    new_theta[var] = val
    return new_theta

# --- Forward Chaining Logic ---

def fol_fc_ask(kb_rules, kb_facts, query):
    current_facts = list(kb_facts)
    var_counter = 1

    print("\n--- Starting Forward Chaining Inference Trace ---")

    # Check if the query already exists initially
    for fact in current_facts:
        phi = unify(fact, query)
        if phi is not False:
            print(f"Goal matched initially with substitution: {phi}")
            return phi

    while True:
        new_sentences = []

        for rule in kb_rules:
            std_rule, var_counter = standardize_variables(rule, var_counter)

            # Match premises recursively against existing facts
            def match_premises(premises, theta):
                if not premises:
                    yield theta
                    return

                first_premise = premises[0]
                rest_premises = premises[1:]

                subst_premise = subst_predicate(theta, first_premise)

                for fact in current_facts:
                    new_theta = unify(subst_premise, fact, theta)
                    if new_theta is not False:
                        yield from match_premises(rest_premises, new_theta)

            for theta in match_premises(std_rule.premises, {}):
                q_prime = subst_predicate(theta, std_rule.conclusion)

                # Check for duplicates
                already_exists = any(unify(q_prime, f, {}) is not False for f in current_facts + new_sentences)

                if not already_exists:
                    new_sentences.append(q_prime)
                    print(f"Inferred: {q_prime[0]}({','.join(q_prime[1])})")

                    phi = unify(q_prime, query)
                    if phi is not False:
                        print(f"Success! Query reached with substitution: {phi}")
                        return phi

        if not new_sentences:
            print("No new facts can be inferred. Search complete.")
            break

        current_facts.extend(new_sentences)

    return False

# --- Parsers ---

def parse_predicate(text):
    match = re.match(r"(\w+)\((.+)\)", text.strip())
    if not match:
        raise ValueError(f"Invalid format: {text}")
    return (match.group(1), [arg.strip() for arg in match.group(2).split(",")])

def parse_rule(text):
    if "=>" not in text:
        raise ValueError("Rules must contain '=>' notation.")
    left, right = text.split("=>")
    premises = [parse_predicate(p) for p in left.split("^") if p.strip()]
    return Rule(premises, parse_predicate(right))

# --- Main Program Entry ---

def main():
    print("=================================================================")
    print("   First-Order Logic Forward Chaining Engine (FOL-FC-ASK)   ")
    print("=================================================================")
    print("Rules formatting rules:\n- Use lowercase single letters for variables: x, y, z")
    print("- Capitalized names are treated as constants: Robert, CountryA")
    print("- Connect conditions with '^' and deductions with '=>'\n")

    facts = []
    print("--- Enter Initial Facts (Leave empty and press Enter to finish) ---")
    while True:
        raw_fact = input("Fact: ").strip()
        if not raw_fact: break
        try:
            facts.append(parse_predicate(raw_fact))
        except Exception as e:
            print(f"Error parsing fact: {e}")

    rules = []
    print("\n--- Enter Rules (Leave empty and press Enter to finish) ---")
    while True:
        raw_rule = input("Rule: ").strip()
        if not raw_rule: break
        try:
            rules.append(parse_rule(raw_rule))
        except Exception as e:
            print(f"Error parsing rule: {e}")

    print("\n--- Enter Target Query ---")
    while True:
        raw_query = input("Query: ").strip()
        if raw_query:
            try:
                query = parse_predicate(raw_query)
                break
            except Exception as e:
                print(f"Error parsing query: {e}")

    result = fol_fc_ask(rules, facts, query)
    print("\n--- Final Summary ---")
    print(f"Result: {True if result is not False else False}")

if __name__ == "__main__":
    main()

   First-Order Logic Forward Chaining Engine (FOL-FC-ASK)   
Rules formatting rules:
- Use lowercase single letters for variables: x, y, z
- Capitalized names are treated as constants: Robert, CountryA
- Connect conditions with '^' and deductions with '=>'

--- Enter Initial Facts (Leave empty and press Enter to finish) ---
Fact: American(Robert)
Fact: Enemy(CountryA)
Fact: Missile(M1)
Fact: Sells(Robert,M1,CountryA)
Fact: 

--- Enter Rules (Leave empty and press Enter to finish) ---
Rule: Missile(x) => Weapon(x)
Rule: Enemy(x) => Hostile(x)
Rule: American(x) ^ Weapon(y) ^ Sells(x,y,z) ^ Hostile(z) => Criminal(x)
Rule: 

--- Enter Target Query ---
Query: Criminal(Robert)

--- Starting Forward Chaining Inference Trace ---
Inferred: Weapon(M1)
Inferred: Hostile(CountryA)
Inferred: Criminal(Robert)
Success! Query reached with substitution: {}

--- Final Summary ---
Result: True
